# Neuro-Genomics Final Project Workflow

This notebook is designed for **Google Colab** and follows the repository instructions.
It has two parts:
1. Bulk RNA-seq differential expression with **DESeq2** (R via rpy2).
2. Single-cell spatial analysis in Python.

> Important: Replace placeholder file names/column names where needed, based on your provided dataset.

## 0) Colab setup

Install Python/R dependencies in Colab, create output folders, and set paths.

In [ ]:
# Colab setup: install required Python packages
!pip -q install rpy2 pandas numpy matplotlib seaborn scikit-learn

# Install R and DESeq2 in Colab (safe to re-run)
!apt-get -qq update
!apt-get -qq install -y r-base r-base-dev

# Install BiocManager + DESeq2
!R -q -e "if (!require('BiocManager', quietly=TRUE)) install.packages('BiocManager', repos='https://cloud.r-project.org'); BiocManager::install('DESeq2', ask=FALSE, update=FALSE)"

In [ ]:
from pathlib import Path
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Output folders
Path('results/part1_bulk').mkdir(parents=True, exist_ok=True)
Path('results/part2_single_cell').mkdir(parents=True, exist_ok=True)

# Google Drive folder configuration (Colab)
# Public shared folder containing all required data files.
DRIVE_FOLDER_URL = 'https://drive.google.com/drive/folders/1ZP0T8l5Qx1TJ5iihL9xCSKmbEc0MdC3r?usp=drive_link'
DATA_DIR = Path('Data')
REQUIRED_FILES = ['expression_matrix.csv', 'locations_of_cells.csv', 'marker_genes.csv']


def _extract_folder_id(url: str) -> str | None:
    m = re.search(r'/folders/([a-zA-Z0-9_-]+)', url)
    return m.group(1) if m else None


try:
    import gdown
except ImportError:
    !pip -q install gdown
    import gdown

DATA_DIR.mkdir(parents=True, exist_ok=True)
missing_files = [f for f in REQUIRED_FILES if not (DATA_DIR / f).exists()]

if missing_files:
    print('Missing files locally, downloading from shared Google Drive folder...')
    # Try URL first
    downloaded = gdown.download_folder(url=DRIVE_FOLDER_URL, output=str(DATA_DIR), quiet=False, use_cookies=False)

    # Fallback: use extracted folder ID if URL mode fails in some Colab sessions
    if downloaded is None:
        folder_id = _extract_folder_id(DRIVE_FOLDER_URL)
        if folder_id:
            gdown.download_folder(id=folder_id, output=str(DATA_DIR), quiet=False, use_cookies=False)

missing_after_download = [f for f in REQUIRED_FILES if not (DATA_DIR / f).exists()]
if missing_after_download:
    raise FileNotFoundError(
        f"Could not find required files in {DATA_DIR}: {missing_after_download}. "
        f"Please verify the shared folder permissions and file names."
    )

print('DATA_DIR:', DATA_DIR.resolve())
print('Files:', sorted([p.name for p in DATA_DIR.glob('*')]))



## Part 1: Bulk RNA-seq (DESeq2 in R via rpy2)

### 1.1 Load count matrix and prepare sample table

Expected format for count matrix:
- Rows = genes
- Columns = samples
- Values = raw integer counts

If your file or column names are different, update the placeholders in the next cell.

In [ ]:
# ---- Bulk RNA-seq input placeholders ----
counts_file = DATA_DIR / 'bulk_counts_matrix.csv'  # TODO: replace with real file name

# Example condition list (same order as count columns). Replace with real conditions.
# Example: ['Control','Control','Treatment','Treatment']
condition_labels = ['Control', 'Control', 'Treatment', 'Treatment']

# Load counts
counts_df = pd.read_csv(counts_file, index_col=0)
print('Counts shape:', counts_df.shape)
counts_df.head()

In [ ]:
# Build sample condition table
sample_table = pd.DataFrame({
    'sample': counts_df.columns,
    'condition': condition_labels
})

# Safety check
assert len(sample_table) == counts_df.shape[1], 'condition_labels length must match number of samples'

sample_table.to_csv('results/part1_bulk/sample_conditions.csv', index=False)
counts_df.to_csv('results/part1_bulk/counts_input_used.csv')
print(sample_table)

### 1.2 Enable rpy2 and move data to R

In [ ]:
%load_ext rpy2.ipython
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
pandas2ri.activate()

In [ ]:
# Pass Python dataframes to R
ro.globalenv['counts_py'] = counts_df
ro.globalenv['sample_table_py'] = sample_table

### 1.3 Run DESeq2 in R

This cell runs DESeq2, exports normalized counts and differential expression results.

In [ ]:
%%R
library(DESeq2)

# Convert Python objects to R data.frames
counts <- as.data.frame(counts_py)
sample_table <- as.data.frame(sample_table_py)

# Ensure row/column compatibility
rownames(sample_table) <- sample_table$sample
counts_mat <- as.matrix(counts)
mode(counts_mat) <- 'integer'

# Reorder sample table to match count matrix columns
sample_table <- sample_table[colnames(counts_mat), , drop=FALSE]

# DESeq2 dataset
dds <- DESeqDataSetFromMatrix(
  countData = counts_mat,
  colData = sample_table,
  design = ~ condition
)

# Run DE analysis
dds <- DESeq(dds)
res <- results(dds)
res_df <- as.data.frame(res)
res_df$gene <- rownames(res_df)

# Save outputs
write.csv(as.data.frame(counts(dds, normalized=TRUE)), 'results/part1_bulk/normalized_counts_deseq2.csv')
write.csv(res_df, 'results/part1_bulk/deseq2_results.csv', row.names=FALSE)


### 1.4 Filter significant genes and split up/down

In [ ]:
res_df = pd.read_csv('results/part1_bulk/deseq2_results.csv')

# Basic significance filter
sig = res_df[(res_df['padj'].notna()) & (res_df['padj'] < 0.05)]
up = sig[sig['log2FoldChange'] > 0]
down = sig[sig['log2FoldChange'] < 0]

sig.to_csv('results/part1_bulk/significant_genes_padj_0.05.csv', index=False)
up.to_csv('results/part1_bulk/upregulated_genes.csv', index=False)
down.to_csv('results/part1_bulk/downregulated_genes.csv', index=False)

print('Significant:', len(sig), '| Up:', len(up), '| Down:', len(down))

### 1.5 Basic plots (PCA, MA-like, volcano)

In [ ]:
# PCA on log-normalized counts
norm_counts = pd.read_csv('results/part1_bulk/normalized_counts_deseq2.csv', index_col=0)
log_norm = np.log2(norm_counts + 1).T

from sklearn.decomposition import PCA
pca = PCA(n_components=2)
pca_coords = pca.fit_transform(log_norm)

pca_df = pd.DataFrame(pca_coords, columns=['PC1','PC2'])
pca_df['sample'] = log_norm.index
pca_df = pca_df.merge(sample_table, on='sample', how='left')

plt.figure(figsize=(6,5))
sns.scatterplot(data=pca_df, x='PC1', y='PC2', hue='condition', s=90)
plt.title('PCA of samples (log2 normalized counts)')
plt.tight_layout()
plt.savefig('results/part1_bulk/pca_samples.png', dpi=150)
plt.show()

In [ ]:
# MA-like plot
plot_df = res_df.copy()
plot_df['baseMean'] = pd.to_numeric(plot_df['baseMean'], errors='coerce')
plot_df['log2FoldChange'] = pd.to_numeric(plot_df['log2FoldChange'], errors='coerce')
plot_df['padj'] = pd.to_numeric(plot_df['padj'], errors='coerce')
plot_df['significant'] = plot_df['padj'] < 0.05

plt.figure(figsize=(6,5))
plt.scatter(np.log10(plot_df['baseMean'] + 1), plot_df['log2FoldChange'],
            c=plot_df['significant'].map({True:'red', False:'gray'}), s=10, alpha=0.6)
plt.axhline(0, color='black', linestyle='--', linewidth=1)
plt.xlabel('log10(baseMean + 1)')
plt.ylabel('log2FoldChange')
plt.title('MA-like plot')
plt.tight_layout()
plt.savefig('results/part1_bulk/ma_plot_like.png', dpi=150)
plt.show()

In [ ]:
# Volcano plot
plot_df['neglog10_padj'] = -np.log10(plot_df['padj'])

plt.figure(figsize=(6,5))
plt.scatter(plot_df['log2FoldChange'], plot_df['neglog10_padj'],
            c=plot_df['significant'].map({True:'crimson', False:'lightgray'}),
            s=10, alpha=0.7)
plt.xlabel('log2FoldChange')
plt.ylabel('-log10(adjusted p-value)')
plt.title('Volcano plot')
plt.tight_layout()
plt.savefig('results/part1_bulk/volcano_plot.png', dpi=150)
plt.show()

## Part 2: Single-cell spatial analysis

### 2.1 Load required files

Required input files:
- `expression_matrix.csv`
- `locations_of_cells.csv`
- `marker_genes.csv`

If your columns differ from placeholders, edit mappings in the next cells.

In [ ]:
expr_file = DATA_DIR / 'expression_matrix.csv'
loc_file = DATA_DIR / 'locations_of_cells.csv'
marker_file = DATA_DIR / 'marker_genes.csv'

expr = pd.read_csv(expr_file, index_col=0)
# Drop empty/unnamed columns that can appear at the end of CSV exports
expr = expr.loc[:, expr.columns.notna()]
expr = expr.loc[:, ~expr.columns.astype(str).str.startswith('Unnamed')]
loc = pd.read_csv(loc_file)
markers = pd.read_csv(marker_file)

print('expr shape:', expr.shape)
print('loc shape:', loc.shape)
print('markers shape:', markers.shape)

expr.head()

In [ ]:
# Columns in provided marker file
marker_celltype_col = 'CellType'
marker_gene_col = 'Marker'

# Build marker dictionary
marker_dict = markers.groupby(marker_celltype_col)[marker_gene_col].apply(list).to_dict()
print('Cell types in marker table:', list(marker_dict.keys()))

### 2.2 Assign likely cell type based on marker genes

In [ ]:
# Score each cell by average expression of marker genes for each candidate cell type
common_genes = set(expr.columns)
cell_scores = pd.DataFrame(index=expr.index)

for ct, genes in marker_dict.items():
    valid = [g for g in genes if g in common_genes]
    if len(valid) == 0:
        cell_scores[ct] = 0
    else:
        cell_scores[ct] = expr[valid].mean(axis=1)

assigned_type = cell_scores.idxmax(axis=1)
assigned_score = cell_scores.max(axis=1)

assign_df = pd.DataFrame({
    'cell_id': expr.index,
    'assigned_cell_type': assigned_type.values,
    'assignment_score': assigned_score.values
})

assign_df.to_csv('results/part2_single_cell/cell_type_assignments.csv', index=False)

In [ ]:
# Define immune/tumor/other categories with beginner-friendly placeholders
# Update immune and tumor labels to match your marker file terminology.
immune_labels = {'immune', 't_cell', 'b_cell', 'macrophage', 'nk_cell'}
tumor_labels = {'tumor', 'malignant', 'cancer_cell'}

def coarse_label(x):
    lx = str(x).lower().strip()
    if lx in immune_labels:
        return 'immune'
    if lx in tumor_labels:
        return 'tumor'
    return 'other'

assign_df['coarse_type'] = assign_df['assigned_cell_type'].map(coarse_label)
immune_pct = 100 * (assign_df['coarse_type'] == 'immune').mean()
print(f'Immune cells: {immune_pct:.2f}%')

assign_df.to_csv('results/part2_single_cell/cell_type_assignments_with_coarse_labels.csv', index=False)
pd.DataFrame({'metric':['immune_percentage'], 'value':[immune_pct]}).to_csv('results/part2_single_cell/immune_percentage.csv', index=False)

### 2.3 Detect PD-L1-positive cells

The official gene symbol for PD-L1 is often `CD274`. If project/course files define a different symbol convention, replace accordingly.

In [ ]:
# Try expected PD-L1 symbols in order
pdl1_candidates = ['CD274', 'Cd274', 'PD-L1', 'PDL1']
found = [g for g in pdl1_candidates if g in expr.columns]

if len(found) == 0:
    raise ValueError('PD-L1 gene symbol not found in expression matrix. Check course/project naming convention.')

pdl1_gene = found[0]
print('Using PD-L1 symbol:', pdl1_gene)

# Simple positivity rule: expression > 0
assign_df['PDL1_expression'] = expr[pdl1_gene].values
assign_df['PDL1_positive'] = assign_df['PDL1_expression'] > 0
pdl1_pct = 100 * assign_df['PDL1_positive'].mean()

assign_df.to_csv('results/part2_single_cell/cell_annotations_with_pdl1.csv', index=False)
pd.DataFrame({'metric':['pdl1_positive_percentage'], 'value':[pdl1_pct], 'gene_symbol_used':[pdl1_gene]}).to_csv('results/part2_single_cell/pdl1_percentage.csv', index=False)
print(f'PD-L1 positive cells: {pdl1_pct:.2f}%')

### 2.4 Spatial plots

In [ ]:
# Columns in provided location file
cell_id_col = 'Cell number'
x_col = 'Location of the middle of cell in X axis (pixels)'
y_col = 'Location of the middle of cell in Y axis (pixels)'

assign_df_plot = assign_df.rename(columns={'cell_id': cell_id_col})
plot_df = loc[[cell_id_col, x_col, y_col]].merge(assign_df_plot, on=cell_id_col, how='inner')
plot_df.to_csv('results/part2_single_cell/spatial_merged_table.csv', index=False)

plt.figure(figsize=(6,5))
sns.scatterplot(data=plot_df, x=x_col, y=y_col, hue='coarse_type', s=8, alpha=0.8)
plt.title('Spatial map: immune vs tumor vs other')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig('results/part2_single_cell/spatial_cell_classes.png', dpi=150)
plt.show()

plt.figure(figsize=(6,5))
sns.scatterplot(data=plot_df, x=x_col, y=y_col, hue='PDL1_positive', s=8, alpha=0.8)
plt.title('Spatial map: PD-L1 positive cells')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig('results/part2_single_cell/spatial_pdl1_positive.png', dpi=150)
plt.show()

## Final note on interpretation

Write biological conclusions only **after** inspecting the computed outputs (DE gene lists, immune %, PD-L1 %, and plots).
Do not claim findings that are not directly supported by these results.